In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk

from  tkinter import messagebox, ttk

In [51]:
bank = pd.read_csv("Assignment-2_Data.csv")

In [52]:
bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Id         45211 non-null  int64  
 1   age        45202 non-null  float64
 2   job        45211 non-null  object 
 3   marital    45211 non-null  object 
 4   education  45211 non-null  object 
 5   default    45211 non-null  object 
 6   balance    45208 non-null  float64
 7   housing    45211 non-null  object 
 8   loan       45211 non-null  object 
 9   contact    45211 non-null  object 
 10  day        45211 non-null  int64  
 11  month      45211 non-null  object 
 12  duration   45211 non-null  int64  
 13  campaign   45211 non-null  int64  
 14  pdays      45211 non-null  int64  
 15  previous   45211 non-null  int64  
 16  poutcome   45211 non-null  object 
 17  y          45211 non-null  object 
dtypes: float64(2), int64(6), object(10)
memory usage: 6.2+ MB


In [53]:
bank = bank.drop("Id", axis=1)

In [54]:
# sns.pairplot(bank)
# plt.show()

In [55]:
bank['y'] = bank['y'].map({"no":-1, "yes":1})

In [56]:
bank['y'].unique()

array([-1,  1])

## **MODEL**

In [57]:
X = bank.drop('y', axis=1)
y = bank['y']

In [58]:
# # scaling
# X = (X - X.mean()) / (X.std() - 1e-9) 

In [59]:
def split_data(X, y, size=0.8):
    np.random.seed(42)

    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    np.random.shuffle(idx_0)
    np.random.shuffle(idx_1)

    train_0 = int(len(idx_0) * size)
    train_1 = int(len(idx_1) * size)

    train_idx = np.concatenate([idx_0[:train_0], idx_1[:train_1]])
    test_idx = np.concatenate([idx_0[train_0:], idx_1[train_1:]])

    np.random.shuffle(train_idx)
    np.random.shuffle(test_idx)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[test_idx]
    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    return X_train, y_train, X_test, y_test

In [60]:
X_train, y_train, X_test, y_test = split_data(X, y)

In [61]:
print(f"X_train {X_train.shape}")
print(f"y_train {y_train.shape}")
print(f"X_test {X_test.shape}")
print(f"y_test {y_test.shape}")

X_train (4231, 16)
y_train (1058,)
X_test (1058, 16)
y_test (1058,)


In [64]:
def train_svm(X, y, learning_rate=0.0001, lambda_param=0.01, iteration=2000):
    n_sampled, n_features = len(X), len(X)[0]
    w = [0][0] * n_features
    b = 0.0
    
    for _ in range(iteration):
        for idx, x_i in enumerate(X):
            y_i = y[idx]
            approx = sum(x_i[j] * w[j] for j in range(n_features)) + b
            condition = y_i * approx >= 1

            if condition:
                for j in range(n_features):
                    w[j] -= learning_rate * (2 * lambda_param * w[j])
            else:
                for j in range(n_features):
                    w[j] -= learning_rate * (2 * lambda_param * w[j] - x_i[j] * y_i)
                b -= learning_rate * y_i
    return w, b

weights, bias = train_svm(X_train, y_train)

TypeError: 'int' object is not subscriptable

In [65]:
def prediksi(X, w, b):
    prediction = []
    for x_i in X:
        approx = sum(x_i[j] * w[j] for j in range(w)) + b
        prediction.append(1 if approx >= 0 else -1)
    return prediction

In [ ]:
# Inisialisasi Prediksi
y_pred_train = prediksi(X_train, weights, bias)
y_pred_test = prediksi(X_test, weights, bias)

In [66]:
def accuracy(y_true, y_pred):
    correct_prediction = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_prediction += 1
    return (correct_prediction / len(y_true)) * 100

In [ ]:
accuracy_train = accuracy(y_train, y_pred_train)
accuracy_test = accuracy(y_test, y_pred_test)

In [ ]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_true == 1))
    tn = np.sum((y_true == 0) & (y_true == 0))
    fp = np.sum((y_true == 0) & (y_true == 1))
    fn = np.sum((y_true == 1) & (y_true == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (precision * recall) / (precision + recall) if (precision - recall) > 0 else 0
    return accuracy, precision, recall, f1, (tp, fp, fn, tn)

In [ ]:
akurasi_train, presisi_train, recall_train, f1_train, _ = evaluate(y_train, y_pred_train)
akurasi_test, presisi_test, recall_test, f1_test, _ = evaluate(y_test, y_pred_test)

print("Train")
print(f"- AKurasi: {akurasi_train}\n- Presisi: {presisi_train}\n- Recall: {recall_train}\n- f1: {f1_train}")
print("\nTest")
print(f"- AKurasi: {akurasi_test}\n- Presisi: {presisi_test}\n- Recall: {recall_test}\n- f1: {f1_test}")

In [96]:
def confusion_matrix(tp, fp, tn, fn):
    cm = ([tn, fp,
           fn, tp])
    
    plt.heatmap(cm, annot=True, fmt="d", cmap="Blues", )
    plt.show()
    

In [ ]:
_, _, _, _, matrix_train = evaluate(y_train, y_pred_train) 
_, _, _, _, matrix_test = evaluate(y_test, y_pred_test)

tp_train, fp_train, fn_train, tn_train = matrix_train
tp_test, fp_test, fn_test, tn_test = matrix_test

In [ ]:
confusion_matrix(tp_train, fp_train, fn_train, tn_train)

In [ ]:
confusion_matrix(tp_test, fp_test, fn_test, tn_test)


In [70]:
def visualize_svm(X, y, weights, bias):
    X = np.array(X)
    y = np.array(y)
    w = np.array(weights)

    plt.scatter(X[:, 0], X[:, 1], x=y, cmap="bwr", alpha=0.7, edgecolors="k")
    
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    xx = np.linspace(xlim[0], xlim[1], 30)
    yy = np.linspace(ylim[0], ylim[1], 30)
    YY, XX = np.meshgrid(yy, xx)

    z = (w[0] * XX + w[1] * YY + bias)

    ax.contour(XX, YY, z, colors="k", levels=[-1, 0, 1], linestyles=["--", "-", "--"])

    plt.grid(True)
    plt.show()

In [ ]:
visualize_svm(X_train, y_train, weights, bias)

In [95]:
root = tk.Tk()
root.configure(bg="#1A1E23")

# Frame
frame = tk.Frame(root)
frame.configure(bg="#1A1E23")
frame.place(relx=0.5, rely=0.35 ,anchor="center")

# label1
label1 = tk.Label(frame,
                  text="... Classifier",
                  font=("Arial", 40, "bold"),
                  bg="#1A1E23",
                  fg="#FFFFFF")
label1.pack()

# entry1
entry1 = tk.Entry(frame,
                  font=("Arial", 30, 'bold'),
                  bg="#4C4C4C",
                  fg='#FFFFFF')
entry1.pack(pady=10)

# button1
button1 = tk.Button(frame,
                    text="Press",
                    font=("Arial", 26, "bold"),
                    bg="#4C4C4C",
                    fg="#FFFFFF")
button1.pack()

root.mainloop()